# University Doctor Reviews Assistant — Full Pipeline (SDAIA AIE-211)

## Part 0 — Setup

In [ ]:
!pip install -q transformers datasets scikit-learn pandas sentence-transformers faiss-cpu matplotlib

import re
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

random.seed(42)

print("GPU available:", torch.cuda.is_available())

## Part 1 — Arabic Normalization (Lab 4)

In [ ]:
def normalize_arabic(text):
    text = re.sub(r"[إأآا]", "ا", text)
    text = re.sub(r"ى", "ي", text)
    text = re.sub(r"ة", "ه", text)
    text = re.sub(r"[\u064B-\u0652]", "", text)
    text = text.translate(str.maketrans("٠١٢٣٤٥٦٧٨٩", "0123456789"))
    text = re.sub(r"\s+", " ", text).strip()
    return text

sample = "أشرح لنا الدكتور المادة بطريقة ممتازة في الجامعة"

print("Before:", sample)
print("After :", normalize_arabic(sample))

## Part 2 — Dataset & Dialect Audit (Lab 4)

In [ ]:
colleges = {
    "computing":   [("البرمجة", "programming"), ("قواعد البيانات", "databases"), ("الذكاء الاصطناعي", "artificial intelligence"), ("الشبكات", "networks")],
    "sciences":    [("الكيمياء", "chemistry"), ("الفيزياء", "physics"), ("الأحياء", "biology"), ("الرياضيات", "mathematics")],
    "business":    [("المحاسبة", "accounting"), ("التسويق", "marketing"), ("الإدارة", "management"), ("الاقتصاد", "economics")],
    "engineering": [("الدوائر الكهربائية", "circuits"), ("الميكانيكا", "mechanics"), ("التصميم الهندسي", "engineering design"), ("الإنشاءات", "structures")],
}

dialect_templates = {
    "najdi": {
        "subject": [
            "الدكتور يشرح {s} حيل زين",
            "اختبارات {s} صعبة شوي بس عدل",
            "واجبات {s} كثيرة بس تفيدنا",
            "ما استفدت شي من محاضرات {s}",
            "متعاون ويجاوب على اسئلتنا في {s} على طول",
            "يجي على وقته ويمشي على منهج {s}",
        ],
        "generic": ["الدكتور حيل زين وما قصر معنا", "ما استفدت شي منه الصراحة"],
    },
    "hijazi": {
        "subject": [
            "الدكتور يشرح {s} حلو قوي",
            "اختبارات {s} صعبة شوية بس عادلة",
            "واجبات {s} كتير بس مفيدة",
            "ما استفدت من محاضرات {s} خالص",
            "دكتور متعاون ويرد على اسئلتنا في {s} دايما",
            "دحين فهمت {s} بعد شرحه",
        ],
        "generic": ["الدكتور حلو قوي وبيشرح كويس", "ما استفدت منه خالص"],
    },
    "sharqi": {
        "subject": [
            "الدكتور يشرح {s} وايد زين",
            "اختبارات {s} صعبة شوي بس منصفة",
            "واجبات {s} وايد بس مفيدة",
            "ما استفدت شي من محاضرات {s} مب زين",
            "الدكتور متعاون وايد ويجاوب على اسئلتنا في {s}",
            "ملتزم بوقت المحاضرة ومنهج {s}",
        ],
        "generic": ["الدكتور وايد زين ويفهمنا", "ما استفدت شي منه مب زين"],
    },
    "white": {
        "subject": [
            "الدكتور يشرح {s} بطريقة واضحة وممتازة",
            "اختبارات {s} صعبة شوي بس عادلة",
            "واجبات {s} كثيرة بس مفيدة",
            "ما استفدت كثير من محاضرات {s}",
            "الدكتور متعاون ويجاوب على اسئلتنا في {s}",
            "ملتزم بوقت المحاضرة ومنهج {s}",
        ],
        "generic": ["الدكتور ممتاز ويشرح بشكل جيد", "لم استفد كثيرا من الدكتور"],
    },
}

doctor_names = ["خالد", "فهد", "سعد", "محمد", "عبدالله", "ناصر", "سلطان", "ماجد", "تركي", "بندر"]

def add_typo(message):
    words = message.split()
    candidates = [i for i, w in enumerate(words) if len(w) >= 4]
    if not candidates:
        return message
    i = random.choice(candidates)
    w = words[i]
    pos = random.randrange(1, len(w) - 1)
    if random.random() < 0.5:
        words[i] = w[:pos] + w[pos + 1:]
    else:
        words[i] = w[:pos] + w[pos + 1] + w[pos] + w[pos + 2:]
    return " ".join(words)

rows = []
doctor_id = 0

for college, subs in colleges.items():
    for _ in range(20):
        doctor_id += 1
        subject_ar, subject_en = random.choice(subs)
        doctor_name = random.choice(doctor_names)
        for _ in range(6):
            dialect = random.choice(list(dialect_templates))
            generic = random.random() < 0.2
            if generic:
                message = random.choice(dialect_templates[dialect]["generic"])
            else:
                message = random.choice(dialect_templates[dialect]["subject"]).format(s=subject_ar)
            if random.random() < 0.4:
                message = "الدكتور " + doctor_name + ": " + message
            if random.random() < 0.3:
                message = add_typo(message)
            rows.append({
                "doctor_id": doctor_id,
                "text": normalize_arabic(message),
                "college": college,
                "dialect": dialect,
                "subject": subject_en,
                "generic": generic,
            })

df = pd.DataFrame(rows)

before = len(df)
df = df.drop_duplicates("text").reset_index(drop=True)
print("Reviews:", before, "→", len(df), "after removing exact duplicates")

import os

if os.path.exists("my_reviews.csv"):
    my_df = pd.read_csv("my_reviews.csv")
    my_df["text"] = my_df["text"].apply(normalize_arabic)
    for column, default in [("subject", "unknown"), ("generic", False)]:
        if column not in my_df.columns:
            my_df[column] = default
    my_df["generic"] = my_df["generic"].astype(bool)
    df = my_df[["doctor_id", "text", "college", "dialect", "subject", "generic"]].drop_duplicates("text").reset_index(drop=True)
    print("Using my_reviews.csv:", len(df), "reviews")

topics = sorted(df["college"].unique())
label_map = {college: i for i, college in enumerate(topics)}
df["label"] = df["college"].map(label_map)

print("\nLabels:", label_map)
print("\nReviews per college:")
print(df["college"].value_counts())
print("\nReviews per dialect:")
print(df["dialect"].value_counts())
print("\nGeneric reviews (no subject mentioned):", int(df["generic"].sum()))
print("\nDialect x College:")
print(pd.crosstab(df["dialect"], df["college"]))

## Part 3 — Leakage-Free Split by Doctor (Lab 2 & Lab 3)

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

splitter = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=42)
train_idx, test_idx = next(splitter.split(df, df["label"], groups=df["doctor_id"]))

train_df = df.iloc[train_idx]
test_df = df.iloc[test_idx]

print("Train:", len(train_df), "| Test:", len(test_df))
print("Shared Doctors:", set(train_df["doctor_id"]) & set(test_df["doctor_id"]))
print("Shared Texts:", len(set(train_df["text"]) & set(test_df["text"])))
print("\nTest colleges:")
print(test_df["college"].value_counts())
print("\nTest dialects:")
print(test_df["dialect"].value_counts())

## Part 4 — Baseline Model (TF-IDF)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report

def slice_scores(preds, test_data):
    y = test_data["label"].values
    scores = {"overall": f1_score(y, preds, average="macro")}
    for d in sorted(test_data["dialect"].unique()):
        mask = (test_data["dialect"] == d).values
        scores[d] = f1_score(y[mask], preds[mask], average="macro")
    return scores

vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4))
X_train = vec.fit_transform(train_df["text"])
X_test = vec.transform(test_df["text"])

clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, train_df["label"])

base_preds = clf.predict(X_test)

print("Baseline accuracy:", round(accuracy_score(test_df["label"], base_preds), 3))
print(pd.Series(slice_scores(base_preds, test_df)).round(3))

## Part 5 — Fine-tuning & Model Bake-off: CAMeLBERT-mix vs CAMeLBERT-DA (Lab 2 & Lab 4)

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer

model_ids = {
    "CAMeLBERT-mix": "CAMeL-Lab/bert-base-arabic-camelbert-mix",
    "CAMeLBERT-DA":  "CAMeL-Lab/bert-base-arabic-camelbert-da",
}

def make_dataset(tokenizer, data):
    enc = dict(tokenizer(data["text"].tolist(), padding=True, truncation=True))
    enc["labels"] = data["label"].tolist()
    return Dataset.from_dict(enc)

def train_college_model(model_name, train_data, epochs=4, batch_size=16, learning_rate=3e-5):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=len(topics))
    args = TrainingArguments(
        output_dir="./results",
        num_train_epochs=epochs,
        per_device_train_batch_size=batch_size,
        learning_rate=learning_rate,
        save_strategy="no",
        report_to="none",
    )
    trainer = Trainer(model=model, args=args, train_dataset=make_dataset(tokenizer, train_data))
    trainer.train()
    return trainer, tokenizer

def predict_proba(trainer, tokenizer, text_list):
    enc = dict(tokenizer(text_list, padding=True, truncation=True))
    output = trainer.predict(Dataset.from_dict(enc))
    logits = output.predictions
    e = np.exp(logits - logits.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)

trained = {}
results = {}

for name, model_name in model_ids.items():
    trainer, tokenizer = train_college_model(model_name, train_df)
    probs = predict_proba(trainer, tokenizer, test_df["text"].tolist())
    preds = probs.argmax(axis=1)
    trained[name] = (trainer, tokenizer, preds, probs)
    results[name] = slice_scores(preds, test_df)

results["TF-IDF baseline"] = slice_scores(base_preds, test_df)

table = pd.DataFrame(results).T.round(3)
print(table)

best_name = table.drop("TF-IDF baseline")["overall"].idxmax()
best_trainer, best_tokenizer, best_preds, best_probs = trained[best_name]

print("\nBest model:", best_name)

## Part 6 — Slot Extraction: Doctor & Subject (Lab 3B idea: word-level labels)

In [ ]:
all_subjects = [(normalize_arabic(ar), en, college) for college, subs in colleges.items() for ar, en in subs]

def extract_slots(text):
    slots = {}
    m = re.search(r"الدكتور (\S+):", text)
    if m:
        slots["doctor"] = m.group(1)
    for ar, en, college in all_subjects:
        if ar in text:
            slots["subject"] = ar
            slots["subject_college"] = college
            break
    return slots

for message in ["الدكتور خالد: اختبارات الكيمياء صعبة شوي بس عدل", "الدكتور حلو قوي وبيشرح كويس"]:
    print(message, "→", extract_slots(normalize_arabic(message)))

## Part 7 — Bilingual Semantic Search over Reviews + Re-ranking (Lab 5)

In [ ]:
import faiss
from sentence_transformers import SentenceTransformer, CrossEncoder

doc_texts = df["text"].tolist()

embedder = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
doc_embeddings = embedder.encode(doc_texts, convert_to_numpy=True).astype("float32")

index = faiss.IndexFlatL2(doc_embeddings.shape[1])
index.add(doc_embeddings)

reranker = CrossEncoder("cross-encoder/mmarco-mMiniLMv2-L12-H384-v1")

def search(query, k=10):
    q = embedder.encode([query], convert_to_numpy=True).astype("float32")
    distances, indices = index.search(q, k)
    return indices[0].tolist()

def search_rerank(query, k=10):
    candidates = search(query, k)
    scores = reranker.predict([[query, doc_texts[i]] for i in candidates])
    order = np.argsort(-np.array(scores))
    return [candidates[i] for i in order]

query_templates = [
    ("هل شرح {ar} واضح؟", "Is the {en} explanation clear?"),
    ("كيف اختبارات {ar}؟", "How are the {en} exams?"),
    ("هل الدكتور متعاون في {ar}؟", "Is the {en} doctor helpful?"),
]

subject_pairs = [(ar, en) for subs in colleges.values() for ar, en in subs]

queries = []

for lang in [0, 1]:
    for ar, en in random.sample(subject_pairs, 16):
        template = random.choice(query_templates)
        queries.append({
            "text": normalize_arabic(template[lang].format(ar=ar, en=en)),
            "lang": "ar" if lang == 0 else "en",
            "relevant": {i for i in range(len(df)) if df["subject"].iloc[i] == en and not df["generic"].iloc[i]},
        })

queries = [q for q in queries if q["relevant"]]
print("Search queries with a known answer:", len(queries))

def retrieval_scores(rank_fn, k=10):
    rows = []
    for q in queries:
        ranked = rank_fn(q["text"], k)
        rank = next((r + 1 for r, i in enumerate(ranked) if i in q["relevant"]), None)
        rows.append({"lang": q["lang"], "recall": 1 if rank else 0, "mrr": 1 / rank if rank else 0})
    return pd.DataFrame(rows)

stage1 = retrieval_scores(search)
stage2 = retrieval_scores(search_rerank)

print(pd.DataFrame({
    "Stage 1 (FAISS)": stage1[["recall", "mrr"]].mean(),
    "Stage 2 (Re-ranked)": stage2[["recall", "mrr"]].mean(),
}).round(3))

print("\nMRR@10 by query language:")
print(pd.DataFrame({
    "Stage 1": stage1.groupby("lang")["mrr"].mean(),
    "Stage 2": stage2.groupby("lang")["mrr"].mean(),
}).round(3))

## Part 8 — Evaluation & Error Analysis (Day 3)

In [ ]:
print(classification_report(test_df["label"], best_preds, target_names=topics, zero_division=0))

print("Confusion matrix (rows = true, columns = predicted):")
print(pd.crosstab(test_df["college"].values, np.array(topics)[best_preds], rownames=["true"], colnames=["pred"]))

errors = test_df.assign(pred=np.array(topics)[best_preds], confidence=best_probs.max(axis=1).round(2))
errors = errors[errors["college"] != errors["pred"]]

print("\nMisclassified reviews:", len(errors))
print("  of which generic (no subject mentioned):", int(errors["generic"].sum()))
print(errors[["text", "college", "pred", "dialect", "generic", "confidence"]].head(15).to_string())

generic_mask = test_df["generic"].values
print("\nAccuracy on reviews that mention a subject:", round((best_preds[~generic_mask] == test_df["label"].values[~generic_mask]).mean(), 3))
if generic_mask.sum() > 0:
    print("Accuracy on generic reviews:", round((best_preds[generic_mask] == test_df["label"].values[generic_mask]).mean(), 3))

threshold = 0.60
confident = best_probs.max(axis=1) >= threshold

print("\nConfidence threshold:", threshold)
print("Marked as unclear:", round(1 - confident.mean(), 3))
if confident.sum() > 0:
    print("Accuracy on confident predictions:", round((best_preds[confident] == test_df["label"].values[confident]).mean(), 3))

per_college = f1_score(test_df["label"], best_preds, average=None, labels=list(range(len(topics))), zero_division=0)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(topics, per_college)
axes[0].set_title("F1 per college (" + best_name + ")")
axes[0].set_ylim(0, 1)
axes[0].tick_params(axis="x", rotation=30)

table[["overall", "najdi", "hijazi", "sharqi", "white"]].T.plot(kind="bar", ax=axes[1])
axes[1].set_title("Macro-F1 per dialect")
axes[1].set_ylim(0, 1)
axes[1].tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.savefig("results_chart.png", dpi=150)
plt.show()

## Part 9 — The Assistant (Full Program)

In [ ]:
def analyze_review(message, threshold=0.60):
    text = normalize_arabic(message)

    probs = predict_proba(best_trainer, best_tokenizer, [text])[0]
    model_college = topics[int(probs.argmax())]
    confidence = float(probs.max())

    slots = extract_slots(text)

    if confidence >= threshold:
        college, source = model_college, "model"
    elif "subject_college" in slots:
        college, source = slots["subject_college"], "subject keyword"
    else:
        college, source = "unclear", "low confidence"

    similar = [doc_texts[i] for i in search_rerank(text, 10)[:3]]

    return {"college": college, "source": source, "confidence": round(confidence, 2),
            "slots": slots, "similar_reviews": similar}

demo_reviews = [
    "الدكتور خالد: اختبارات الكيمياء صعبة شوي بس عدل",
    "ما استفدت شي من محاضرات قواعد البيانات",
    "دكتور متعاون ويرد على اسئلتنا في الاقتصاد دايما",
    "الدكتور حلو قوي وبيشرح كويس",
]

for review in demo_reviews:
    result = analyze_review(review)
    print("Review  :", review)
    print("College :", result["college"], "(" + result["source"] + ") | confidence:", result["confidence"], "| slots:", result["slots"])
    print("Similar :")
    for s in result["similar_reviews"]:
        print("   -", s)
    print()

## Part 10 — Summary for the Presentation

In [ ]:
table.to_csv("results_table.csv")

print(table)
print("\nBest model:", best_name)
print("Best model macro-F1:", table.loc[best_name, "overall"])
print("TF-IDF baseline macro-F1:", table.loc["TF-IDF baseline", "overall"])
print("Search MRR@10 (Stage 1 / Stage 2):", round(stage1["mrr"].mean(), 3), "/", round(stage2["mrr"].mean(), 3))
print("Accuracy on generic reviews:", round((best_preds[generic_mask] == test_df["label"].values[generic_mask]).mean(), 3) if generic_mask.sum() > 0 else "n/a")